# 1. Initialization

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

spark = SparkSession.builder.getOrCreate()

try:
    dbutils.widgets.text("batch_id", "")
    batch_id = dbutils.widgets.get("batch_id")
except:
    batch_id = None

print("[INFO] Products Silver Pipeline Started")

# 2. Read Bronze products data

In [ ]:
bronze_path = "/Volumes/datalake_catalog/datalake_schema/bronze/products"

df_bronze = (
    spark.read.format("delta").load(bronze_path)
)

df_bronze.show(5)

# 3. Drop control columns

In [ ]:
df1 = df_bronze.drop("id", "source_identifier", "batch_id")
df1.show(5)

# 4. Enforce grain = 1 row per product

In [ ]:
df_clean = (
    df1
    .withColumn("created_at", F.to_timestamp("created_at"))
    .withColumn("ingest_time", F.to_timestamp("ingest_time"))
)

w = Window.partitionBy("product_id", "created_at").orderBy(F.col("ingest_time").desc())

df2 = (
    df_clean
    .withColumn("rn", F.row_number().over(w))
    .filter("rn = 1")
    .drop("rn")
)

df_quarantine_dup = (
    df_clean
    .withColumn("rn", F.row_number().over(w))
    .filter("rn > 1")
    .drop("rn")
)

# 5. Type enforcement

In [ ]:
df3 = (
    df2
    .withColumn("product_id", F.col("product_id").cast("int"))
    .withColumn("product_name", F.col("product_name").cast("string"))
    .withColumn("category", F.col("category").cast("string"))
    .withColumn("created_at", F.col("created_at").cast("timestamp"))
    .withColumn("ingest_time", F.col("ingest_time").cast("timestamp"))
)

# 6. Null validation

In [ ]:
df4 = df3.filter(
    F.col("product_id").isNotNull() &
    F.col("product_name").isNotNull() &
    F.col("category").isNotNull() &
    F.col("ingest_time").isNotNull()
)

df_quarantine_null = df3.subtract(df4)

df4.show(10)

# 7. Clean text fields

In [ ]:
df5 = df4.withColumn("product_name", F.trim(F.col("product_name")))

# 8. Silver upsert (Delta)

In [ ]:
silver_path = "/Volumes/datalake_catalog/datalake_schema/silver/products"

df_upsert = df5

w = Window.partitionBy("product_id").orderBy(
    F.col("ingest_time").desc(), F.col("created_at").desc()
)

df_upsert = (
    df_upsert
    .withColumn("rn", F.row_number().over(w))
    .filter("rn = 1")
    .drop("rn")
)

silver_cols = df_upsert.columns

if DeltaTable.isDeltaTable(spark, silver_path):
    print("[INFO] Updating existing Silver table")
    target = DeltaTable.forPath(spark, silver_path)

    target.alias("t").merge(
        df_upsert.alias("s"),
        "t.product_id = s.product_id"
    ).whenMatchedUpdate(
        condition="s.ingest_time > t.ingest_time",
        set={c: f"s.{c}" for c in silver_cols}
    ).whenNotMatchedInsert(
        values={c: f"s.{c}" for c in silver_cols}
    ).execute()

else:
    print("[INFO] Creating Silver table")
    df_upsert.write.format("delta").mode("overwrite").save(silver_path)

# 9. Validate output

In [ ]:
df_check = spark.read.format("delta").load(silver_path)
df_check.show(10)

# 10. Completion

In [ ]:
print("[DONE] Products Silver Pipeline Completed")